# Analyzing Gravitas data

You exported a CSV from [Gravitas](https://gravitas-sim.online). This notebook reads it
and does the four things the data is good for:

1. **Draw the orbits** — check that the file is what you think it is.
2. **Measure a period** — from the data, not from a formula.
3. **Test Kepler's third law** — plot $P^2$ against $a^3$ and fit the slope.
4. **Check energy conservation** — is the integrator behaving?

And, if you exported a light curve, a fifth: measure a transit depth and turn it into a planet radius.

Nothing here needs installing. Run each cell in order with `Shift + Enter`.

---

## Getting your file in

In Gravitas, press **E** (or **Export Data** in the right-hand rail) and download
`...-trajectories.csv`. Then run the cell below and choose it.

If you are not in Colab, replace the whole cell with:

```python
traj = pd.read_csv('gravitas-solar-system-trajectories.csv')
```


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import files
    uploaded = files.upload()          # pick your trajectories CSV
    name = next(iter(uploaded))
    traj = pd.read_csv(io.BytesIO(uploaded[name]))
except ImportError:
    # Not running in Colab: point this at the file you downloaded.
    traj = pd.read_csv('gravitas-trajectories.csv')

print(f'{len(traj):,} rows, {traj.name.nunique()} objects')
traj.head()


## What is in the file

One row per object per recorded frame. Every column carries its unit in the name,
so there is nothing to look up:

| column | meaning |
|---|---|
| `t_days` | simulation clock, in days |
| `name`, `type`, `object_id` | which object this row is |
| `mass_msun` | its mass, in solar masses |
| `x_au`, `y_au` | position, in astronomical units |
| `vx_kms`, `vy_kms`, `speed_kms` | velocity, in km/s |
| `primary` | what it is orbiting |
| `r_au` | its distance from that primary |
| `theta_deg` | its angle around that primary |
| `E_kin_J`, `E_pot_J`, `E_tot_J` | energies relative to the primary, in joules |

`r_au`, `theta_deg` and the energies are all measured **relative to the primary**, which
is why a whole system drifting across the screen does not look unbound, and why you can
measure a period without first working out where the center of the orbit is.

An object with nothing to orbit leaves those columns empty rather than zero.


In [ ]:
print(traj.groupby(['name', 'type', 'primary'])
      .agg(rows=('t_days', 'size'),
           mass_msun=('mass_msun', 'first'),
           r_min_au=('r_au', 'min'),
           r_max_au=('r_au', 'max'))
      .round(4))


## 1. Draw the orbits

The first thing to do with any data set is look at it.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

for name, g in traj.groupby('name'):
    ax.plot(g.x_au, g.y_au, lw=1, label=name)
    ax.plot(g.x_au.iloc[-1], g.y_au.iloc[-1], 'o', ms=4)

ax.set_xlabel('x (AU)')
ax.set_ylabel('y (AU)')
ax.set_aspect('equal')
ax.set_title('Recorded orbits')
ax.legend(fontsize=8, loc='upper right')
ax.grid(alpha=0.2)
plt.show()


## 2. Measure a period

The period is how long the object takes to go once round. So: add up the angle it turned
through, divide by 360 degrees to get the number of laps, and divide the elapsed time by
that.

Two details make this work on real recordings rather than only on tidy ones.

**Gaps.** If you paused, scrubbed the timeline or left the tab in the background, the
recording has a jump in it. Stepping across a jump would count one lap where twenty
happened. So each step is checked against the typical spacing, and anything much larger
is dropped: both its angle and its time, together, so the two stay consistent.

**Wrapping.** An angle that goes past 180 degrees comes back as negative. Each step is
therefore folded into the range from -180 to +180, which is correct as long as the object
does not travel more than half a lap between samples. If it does, the function says so
instead of quietly returning a wrong answer.


In [ ]:
def measure_period(g):
    """Orbital period in days, from the angle swept around the primary.

    Returns (period, laps, warning). Period is NaN if there is not enough to
    go on, and warning is a string when the answer should not be trusted.
    """
    g = g.sort_values('t_days')
    t = g.t_days.to_numpy(float)
    th = np.radians(g.theta_deg.to_numpy(float))
    good = np.isfinite(t) & np.isfinite(th)
    t, th = t[good], th[good]
    if len(t) < 3:
        return np.nan, 0.0, 'fewer than three usable samples'

    dt = np.diff(t)
    # Fold each step into -180..180: the shortest way round is the one that
    # happened, as long as the sampling keeps up with the orbit.
    dth = (np.diff(th) + np.pi) % (2 * np.pi) - np.pi

    # Drop steps across a gap in the recording, and any step of zero length.
    typical = np.median(dt[dt > 0]) if np.any(dt > 0) else 0.0
    keep = (dt > 0) & (dt < 4 * typical)
    if not keep.any():
        return np.nan, 0.0, 'no usable time steps'
    dropped = int((~keep).sum())

    laps = float(abs(dth[keep].sum()) / (2 * np.pi))
    span = float(dt[keep].sum())
    if laps < 1e-9:
        return np.nan, 0.0, 'it has barely moved'

    warn = ''
    biggest = float(np.abs(dth[keep]).max())
    if biggest > 0.6 * np.pi:
        # More than about a third of a lap between samples: the fold above can
        # no longer tell forwards from backwards.
        warn = 'sampled too coarsely for this orbit'
    elif dropped:
        warn = f'{dropped} step(s) skipped across a gap in the recording'

    return span / laps, laps, warn


periods = {}
for name, g in traj.groupby('name'):
    P, laps, warn = measure_period(g)
    periods[name] = P
    if np.isnan(P):
        print(f'{name:>22}:  no period  ({warn})')
    else:
        note = f'   [{warn}]' if warn else ''
        print(f'{name:>22}:  P = {P:10.2f} days   ({laps:6.2f} laps recorded){note}')


> **A lap count well under 1** means the object has barely started its orbit, and the
> period is an extrapolation from a short arc rather than a measurement. It will still be
> roughly right for a circular orbit and can be badly wrong for an eccentric one. To fix
> it, go back to Gravitas, speed the simulation up with the **Fast** button, let it run
> until the object has gone round a few times, and export again.


## 3. Test Kepler's third law

Kepler found that $P^2 \propto a^3$. Newton showed the constant of proportionality
depends on the mass being orbited, which is what makes an orbit a way of *weighing*
something:

$$a^3 = P^2 (M_1 + M_2)$$

with $a$ in AU, $P$ in years and masses in $M_\odot$. For a planet around a star the
planet's mass is negligible, so the slope of $a^3$ against $P^2$ **is the mass of the star**.

The semi-major axis $a$ is the average of the closest and furthest distances.


In [ ]:
DAYS_PER_YEAR = 365.25

rows = []
for name, g in traj.groupby('name'):
    P_days, laps, _ = measure_period(g)
    if np.isnan(P_days):
        continue
    r_min, r_max = g.r_au.min(), g.r_au.max()
    rows.append({
        'name': name,
        'primary': g.primary.iloc[0],
        'a_au': 0.5 * (r_min + r_max),
        'e': (r_max - r_min) / (r_max + r_min),
        'P_yr': P_days / DAYS_PER_YEAR,
        'laps': laps,
        'mass_msun': g.mass_msun.iloc[0],
    })

orbits = pd.DataFrame(rows).sort_values('a_au').reset_index(drop=True)
orbits['a_cubed'] = orbits.a_au ** 3
orbits['P_squared'] = orbits.P_yr ** 2
orbits.round(5)


In [ ]:
# Objects that have not gone round at least a fifth of a lap are extrapolations,
# not measurements, so they are left out of the fit.
fit = orbits[orbits.laps >= 0.2]

if len(fit) < 2:
    print('Need at least two objects with a measured period to fit a line.')
    print('Let the simulation run for longer and export again.')
else:
    # Forced through the origin: an orbit of no size takes no time, so the
    # relationship has no constant term to fit.
    x, y = fit.P_squared.values, fit.a_cubed.values
    slope = float(np.sum(x * y) / np.sum(x * x))

    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(x, y, zorder=3)
    for _, r in fit.iterrows():
        ax.annotate(r['name'], (r.P_squared, r.a_cubed),
                    textcoords='offset points', xytext=(6, -3), fontsize=8)
    grid = np.linspace(0, x.max() * 1.05, 50)
    ax.plot(grid, slope * grid, '--', lw=1.2,
            label=f'slope = {slope:.3f} $M_\\odot$')
    ax.set_xlabel(r'$P^2$  (years$^2$)')
    ax.set_ylabel(r'$a^3$  (AU$^3$)')
    ax.set_title("Kepler's third law, from your own measurements")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.show()

    print(f'Fitted central mass: {slope:.4f} solar masses')
    print(f'  from {len(fit)} objects: ' + ', '.join(fit.name))

    # The primary's own recorded mass, when it is in the file to compare with.
    known = traj[traj.name.isin(fit.primary)].groupby('name').mass_msun.first()
    if len(known):
        print('\nRecorded mass of each primary:')
        print(known.round(4).to_string())
    else:
        primary = fit.primary.iloc[0]
        print(f'\n{primary!r} is not in this file, so there is nothing to check')
        print('against. Export with "Every object" selected to include it.')


**What to look at.** If the points fall on a straight line through the origin, you have
reproduced Kepler's third law from data you generated yourself. The slope is the mass of
the object everything is going round, in solar masses — compare it with the recorded
mass printed underneath. Agreement to a few percent is what you should expect; the
difference comes from measuring $a$ from a finite number of samples.

---

## 4. Is the integrator behaving?

Total energy should be constant for an isolated pair. Watching it drift is how you find
out whether a numerical result can be trusted, and it is a habit worth building early.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for name, g in traj.groupby('name'):
    e = g.E_tot_J.values
    if not np.isfinite(e).any() or e[0] == 0:
        continue
    ax.plot(g.t_days, (e - e[0]) / abs(e[0]) * 100, lw=1, label=name)

ax.axhline(0, color='k', lw=0.8, alpha=0.4)
ax.set_xlabel('time (days)')
ax.set_ylabel('drift in total energy (%)')
ax.set_title('Energy conservation')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
plt.show()

for name, g in traj.groupby('name'):
    e = g.E_tot_J.values
    if not np.isfinite(e).any() or e[0] == 0:
        continue
    drift = abs(e[-1] - e[0]) / abs(e[0]) * 100
    bound = 'bound' if e[-1] < 0 else 'UNBOUND: it is leaving'
    print(f'{name:>22}:  {drift:7.3f}% drift   {bound}')


The **sign** of the total energy is the whole story about whether something comes back.
Negative means bound and it stays. Positive means it escapes and never returns. Try
exporting the *Interstellar Visitor* scenario and looking at this column for ʻOumuamua.

---

## 5. The light curve

Export `...-lightcurve.csv` and `...-transits.csv` from the same dialog, with the Light
Curve tool running. Upload them below.


In [ ]:
curve = transits = None
try:
    from google.colab import files
    uploaded = files.upload()   # pick the lightcurve CSV, and the transits CSV if you have it
    for key, blob in uploaded.items():
        if 'lightcurve' in key:
            curve = pd.read_csv(io.BytesIO(blob))
        elif 'transits' in key:
            transits = pd.read_csv(io.BytesIO(blob))
except ImportError:
    import os
    for path, target in [('gravitas-lightcurve.csv', 'curve'),
                         ('gravitas-transits.csv', 'transits')]:
        if os.path.exists(path):
            globals()[target] = pd.read_csv(path)

if curve is not None:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(curve.t_days, curve.flux_relative, lw=0.8, color='#888')
    dips = curve[curve.in_transit == 1]
    ax.plot(dips.t_days, dips.flux_relative, '.', ms=3, color='#d05a2a',
            label='in transit')
    ax.set_xlabel('time (days)')
    ax.set_ylabel('relative brightness')
    ax.set_title('Light curve')
    ax.legend()
    ax.grid(alpha=0.2)
    plt.show()
else:
    print('No light curve loaded yet.')


### Depth to planet radius

A planet crossing its star blocks a fraction of the light equal to the ratio of their
areas, so

$$\frac{R_\mathrm{planet}}{R_\star} = \sqrt{\text{depth}}$$

and the spacing between successive transits is the orbital period.

One honest caveat, and it is the subject of a whole section of the *Finding Planets by
Their Shadows* investigation: a star is not uniformly bright. It is brighter at the
center of its disk than at the edge, so a planet crossing the middle blocks more than its
fair share of the light and the dip comes out deeper than the area ratio alone would
give. Taking the square root of the raw depth therefore *overestimates* the planet by
several percent. It is the right first estimate, and correcting it is the next thing to
learn.


In [ ]:
R_SUN_IN_EARTHS = 109.076
R_SUN_IN_JUPITERS = 9.7311

if 'transits' in globals() and transits is not None and len(transits):
    depth = transits.depth_relative.mean()
    ratio = np.sqrt(depth)
    print(f'transits measured:   {len(transits)}')
    print(f'mean depth:          {depth * 100:.3f}%')
    print(f'R_planet / R_star:   {ratio:.4f}')
    print(f'  = {ratio * R_SUN_IN_EARTHS:6.2f} Earth radii, for a Sun-sized star')
    print(f'  = {ratio * R_SUN_IN_JUPITERS:6.2f} Jupiter radii, for a Sun-sized star')
    if len(transits) > 1:
        gaps = np.diff(np.sort(transits.mid_days.values))
        print(f'\nperiod from transit spacing: {gaps.mean():.4f} days'
              f'  (scatter {gaps.std():.4f})')
else:
    print('No transit table loaded. Run the cell above and pick the transits CSV,')
    print('or let the Light Curve run until a dip has completely finished and')
    print('export again.')


---

## Where to take this

Some things worth trying, roughly in order of difficulty:

- Export the **Solar System** scenario and check that the fitted slope comes out at
  1 solar mass. It should, and if it does not, work out why.
- Export **TRAPPIST-1** and fit its central mass. It is a red dwarf, so you should get
  something well under 1 $M_\odot$.
- The `e` column in the `orbits` table is the eccentricity, worked out from the closest
  and furthest distances. Which of your objects is on the least circular orbit?
- Confirm Kepler's *second* law: along one orbit, $r^2 \, d\theta/dt$ should be constant.
  You have `r_au` and `theta_deg`, so this is two lines.
- Export a **binary star** and fit both stars separately. The ratio of their orbit sizes
  is the inverse ratio of their masses.
- Take the same scenario at two different simulation speeds and compare the energy
  drift. Larger time steps cost accuracy, and this is how you see the price.
